<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/notebooks/stage_07_model_training/stage_07_00_model_training_seq2seq_plan.ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Stage_07 – Model Training (Seq2Seq) – Objetivo y Modelos**


## **1. Objetivo de predicción (definición formal)**


Entrenaremos modelos **seq2seq** para predecir una **secuencia futura completa** a partir de una ventana histórica intradía.

- **Entrada (X):** bloque de tamaño **29 × 8**  
  $$
  X \in \mathbb{R}^{29 \times 8}
  $$
  donde 29 = minutos consecutivos (ventana histórica) y 8 = features por minuto.

- **Salida / Target (Y):** bloque de tamaño **29 × 1**  
  $$
  Y \in \mathbb{R}^{29 \times 1}
  $$
  donde 29 = minutos futuros a predecir y 1 = variable objetivo escalar por minuto.

**Interpretación:** el modelo no predice un único valor final, sino la **trayectoria temporal futura completa** (29 pasos).





## **2. Qué NO es este problema**


Quedan explícitamente fuera de alcance:

- **Many-to-one:** predecir solo el último paso \(t+H\).
- **Targets agregados:** suma/promedio/retorno acumulado colapsado en un escalar.
- **Heads independientes por horizonte:** entrenar 29 regresiones separadas sin estructura temporal.


## **3. Modelos a evaluar (comparación escalonada)**


Evaluaremos un set mínimo y robusto, desde baselines hasta modelos fuertes:

### **(A) Baselines obligatorios**


1. **Naive / Persistence**
   - Predice “sin cambio” (por ejemplo, replica el último valor observado) en los 29 pasos.
   - Define el “piso” mínimo de performance.
   - Notebook: `stage_07_01_naive_model.ipynb`

2. **MLP (Direct Multi-step)**
   - Aplana $(29 \times 8)$ y predice $(29 \times 1)$.
   - Base neural simple y rápida para validar el pipeline.
   - Notebook: `stage_07_02_mlp_direct_multistep.ipynb`

### **(B) Seq2Seq clásicos**


3. **Encoder–Decoder GRU**
    - Notebook: `stage_07_03_gru_seq2seq.ipynb`

4. **Encoder–Decoder LSTM**
    - Encoder resume la historia; decoder genera la secuencia futura.
    - Estables y comparables para intradía.
    - Notebook: `stage_07_04_lstm_seq2seq.ipynb`



### **(C) Alternativa robusta no recurrente**


5. **TCN (Temporal Convolutional Network)**
   - Convoluciones causales dilatadas.
   - Buena relación performance/estabilidad.
   - Notebook: `stage_07_05_tcn_seq2seq.ipynb`

### **(D) Modelo de atención**


6. **Transformer Seq2Seq**
   - Encoder–decoder con atención.
   - Requiere regularización y control cuidadoso, pero es candidato fuerte.
   - Notebook: `stage_07_06_transformer_seq2seq.ipynb`

### **(E) Modelo avanzado con estructura temporal explícita**


7. **Temporal Fusion Transformer (TFT)**

- Modelo seq2seq con atención para series temporales multivariadas.
- Incorpora selección de variables y atención temporal para capturar dependencias pasadas y futuras.
- Alta capacidad para modelar patrones intradía complejos; referencia avanzada frente a LSTM, TCN y Transformer estándar.
- Notebook: `stage_07_07_tft_seq2seq.ipynb`

## **4. Definición de la comparación Predicción vs Valor Real (Seq2Seq)**


Para cada muestra del dataset, la evaluación del modelo se define de la siguiente manera:

### **4.1 Esquema de predicción**

- El modelo recibe un **bloque de entrada**:

  $$
  X_t \in \mathbb{R}^{29 \times 8}
  $$

  correspondiente a 29 minutos históricos y 8 features por minuto.

- A partir de ese bloque, el modelo **predice una secuencia futura completa**:

  $$
  \hat{Y}_{t+1:t+29} \in \mathbb{R}^{29 \times 1}
  $$

  es decir, un valor del target por cada uno de los 29 minutos futuros.




### **4.2 Comparación contra el valor real**


La predicción se compara directamente contra la **secuencia real futura observada**:

$$
Y_{t+1:t+29}^{real} \in \mathbb{R}^{29 \times 1}
$$

La comparación es siempre **secuencia contra secuencia**, sin colapsar el target ni aplicar agregaciones previas.




### **4.3 Cálculo de métricas**

A partir de esta comparación base, las métricas se calculan:

- **Por paso temporal**:  
  comparación entre $\hat{y}_{t+k}$ y $y_{t+k}^{real}$, para $(k = 1,\dots,29)$

- **Sobre la trayectoria completa**:  
  error global entre $\hat{Y}_{t+1:t+29}$ y $Y_{t+1:t+29}^{real}$.

No se compara contra un escalar ni contra un valor agregado final.  
El problema es estrictamente **seq2seq**.

### **4.4 Derivación de métricas**


Desde esta comparación fundamental se derivan:

- **Métricas de machine learning**: MAE, RMSE, métricas direccionales.
- **Métricas económicas**: EV, TP/SL, drawdown, usando el **delta real observado** dentro de la ventana futura.

## **5. Métricas de predicción (Machine Learning)**

Aunque el modelo predice una secuencia de 29 pasos, se definen pocas métricas finales, manteniendo la posibilidad de analizar el error por paso sin multiplicar indicadores.



### **5.1 Métricas de error (seq2seq)**

La secuencia predicha $\hat{Y}_{t+1:t+29}$ se compara directamente contra la secuencia real futura $Y_{t+1:t+29}$.

Se reportan las siguientes métricas globales:

- **MAE**  
  Error absoluto medio calculado sobre **todos los pasos de la secuencia**.

- **RMSE**  
  Raíz del error cuadrático medio calculada sobre **todos los pasos de la secuencia**.

> Conceptualmente, los 29 pasos se concatenan y se calcula un único MAE y RMSE por modelo.


### **5.2 Análisis por paso (diagnóstico, no ranking)**


Para **análisis interno y diagnóstico**, el error puede descomponerse por horizonte temporal:

- **MAE(k)** para $k = 1, \dots, 29$

Este análisis:
- permite observar la degradación del error con el horizonte,
- **no se utiliza para ranking ni selección de modelos**.

El ranking entre modelos se realiza **exclusivamente** con MAE y RMSE globales.


### **5.3 Métrica direccional**


- **Directional Accuracy (DA)**  

  Proporción de casos en los que el **signo del delta predicho** coincide con el **signo del delta real**.

  Se define de forma única y simple como:

  - **DA_last**: coincidencia de signo en el **último paso de la secuencia** $k = 29$.

> Esta métrica conecta directamente con la dirección del movimiento esperado a $h$ minutos.



### **5.4 Coeficiente de determinación**


- **$R^2$**  
  Calculado sobre la secuencia completa como métrica estadística complementaria.

### 5.5. **Función de cálculo de métricas**

In [ ]:
import numpy as np
from sklearn.metrics import r2_score

def compute_seq2seq_metrics(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    compute_r2: bool = True,
) -> dict:
    """
    Calcula métricas simples y comparables para modelos seq2seq.

    Parámetros
    ----------
    y_true : np.ndarray
        Valores reales con shape (n_samples, seq_len)
        o (n_samples, seq_len, 1)
    y_pred : np.ndarray
        Valores predichos con shape (n_samples, seq_len)
        o (n_samples, seq_len, 1)
    compute_r2 : bool
        Indica si se debe calcular R² sobre la secuencia completa

    Retorna
    -------
    metrics : dict
        Diccionario con métricas globales y diagnóstico por paso
    """

    # ------------------------------------------------------------------
    # 1) Asegurar que las entradas sean arrays NumPy
    # ------------------------------------------------------------------

    # Convierte y_true a np.ndarray (por si viene como lista o tensor)
    y_true = np.asarray(y_true)

    # Convierte y_pred a np.ndarray
    y_pred = np.asarray(y_pred)

    # ------------------------------------------------------------------
    # 2) Normalizar dimensiones a (n_samples, seq_len)
    # ------------------------------------------------------------------

    # Si y_true tiene dimensión extra (…, 1), la elimina
    if y_true.ndim == 3:
        y_true = y_true.squeeze(-1)

    # Si y_pred tiene dimensión extra (…, 1), la elimina
    if y_pred.ndim == 3:
        y_pred = y_pred.squeeze(-1)

    # Verifica que ambas matrices tengan exactamente el mismo shape
    assert y_true.shape == y_pred.shape, (
        "y_true y y_pred deben tener el mismo shape"
    )

    # Extrae número de muestras y longitud de la secuencia
    n_samples, seq_len = y_true.shape

    # ------------------------------------------------------------------
    # 3) Cálculo de errores
    # ------------------------------------------------------------------

    # Error firmado: diferencia entre predicción y valor real
    errors = y_pred - y_true

    # Error absoluto
    abs_errors = np.abs(errors)

    # ------------------------------------------------------------------
    # 4) Métricas globales (sobre toda la secuencia)
    # ------------------------------------------------------------------

    # MAE global: promedio del error absoluto en todos los pasos
    mae = abs_errors.mean()

    # RMSE global: raíz del promedio del error cuadrático
    rmse = np.sqrt((errors ** 2).mean())

    # ------------------------------------------------------------------
    # 5) Métrica direccional (último paso de la secuencia)
    # ------------------------------------------------------------------

    # Extrae el valor real del último paso (k = seq_len)
    y_true_last = y_true[:, -1]

    # Extrae el valor predicho del último paso
    y_pred_last = y_pred[:, -1]

    # Calcula la accuracy direccional:
    # compara si el signo del delta predicho coincide con el real
    da_last = np.mean(
        np.sign(y_true_last) == np.sign(y_pred_last)
    )

    # ------------------------------------------------------------------
    # 6) MAE por paso (solo diagnóstico)
    # ------------------------------------------------------------------

    # Calcula MAE para cada paso temporal k = 1..seq_len
    mae_per_step = abs_errors.mean(axis=0)  # shape: (seq_len,)

    # ------------------------------------------------------------------
    # 7) Construcción del diccionario de métricas
    # ------------------------------------------------------------------

    metrics = {
        # Error absoluto medio global
        "MAE": float(mae),

        # Raíz del error cuadrático medio global
        "RMSE": float(rmse),

        # Accuracy direccional en el último paso
        "DA_last": float(da_last),

        # MAE por paso (lista para serializar a JSON)
        "MAE_per_step": mae_per_step.tolist(),
    }

    # ------------------------------------------------------------------
    # 8) R² opcional (sobre toda la secuencia concatenada)
    # ------------------------------------------------------------------

    if compute_r2:
        metrics["R2"] = float(
            r2_score(
                y_true.flatten(),   # vectoriza la secuencia real
                y_pred.flatten(),   # vectoriza la secuencia predicha
            )
        )

    # ------------------------------------------------------------------
    # 9) Retorno final
    # ------------------------------------------------------------------

    return metrics

In [ ]:
#Ejemplo de uso:
metrics = compute_seq2seq_metrics(y_true, y_pred)
print(metrics["MAE"], metrics["RMSE"], metrics["DA_last"])

## **6. Interpretación económica de la predicción (ventana de gestación)**


Las predicciones de este Stage_07 se realizan **exclusivamente dentro de la ventana de gestación**
**08:21 – 08:49**.  
En esta ventana **no se ejecutan operaciones reales**. Su objetivo es **anticipar el pre-movimiento
del mercado** antes del inicio de la fase operativa.

Por lo tanto, el modelo no se evalúa como un sistema de trading directo, sino como un **predictor de contexto y filtro de oportunidad**.

### **6.1 Qué está prediciendo realmente el modelo**


Para cada minuto $t$ dentro de la ventana de gestación, el modelo predice un valor:

$$
\Delta pts_h = close_{t+h} - close_t
$$

donde $h$ es el horizonte fijo (60 o 90 minutos).

Cada fila de la salida $29 \times 1$ responde a la pregunta:

> *“Si el mercado estuviera en $t$, ¿cuántos puntos se movería hacia adelante en $h$ minutos?”*

Esto describe el **potencial de movimiento futuro**, no una ganancia inmediata.

### **6.2 Por qué no se usa PnL real en esta ventana**


Dado que:
- no se ejecutan trades en 08:21–08:49, y
- el modelo actúa antes del momento de entrada,

**no tiene sentido evaluar estas predicciones como PnL realizado en esa ventana**.

En cambio, su valor económico se mide por su capacidad de:
- anticipar **movimientos relevantes**, y
- filtrar **jornadas o minutos con oportunidad operativa posterior**.

### **6.3 Métricas económicas como “filtro de oportunidad”**


Se define un umbral económico $Delta_{op}$ (por ejemplo, el *delta_target_p70* del stage_03a), que representa un movimiento considerado **explotable**.

A partir de esto, se evalúa:

- **Opportunity Recall**  
  
  De todos los casos donde el movimiento real fue significativo $|\Delta pts_h| \ge \Delta_{op}$, ¿cuántos fueron correctamente anticipados por el modelo
  (magnitud y signo)?

- **Precision**  
  
  De todas las alertas del modelo, ¿cuántas correspondían realmente a movimientos explotables?

- **Coverage**  
  Proporción de minutos/jornadas marcados como “oportunidad”.  
  Controla si el modelo es demasiado conservador o sobrerreacciona.

Estas métricas evalúan la **utilidad económica del modelo como filtro**, no como ejecutor de trades.


### **6.4 EV “virtual” como métrica comparativa**


Adicionalmente, puede calcularse un **valor esperado teórico (EV)**:

$$
pnl_t = s_t \cdot \Delta pts_h - cost
$$

donde:
- $s_t$ es la señal derivada de la predicción,
- $cost$ representa costos operativos fijos.

Este EV **no representa una ganancia real**, sino un **proxy económico** que permite:
- comparar modelos bajo una misma regla,
- rankearlos de forma consistente,
- evaluar la calidad del setup predictivo.

En resumen, en la ventana de gestación el modelo se evalúa como un **predictor de pre-movimiento y filtro de oportunidad**, y las métricas económicas se interpretan como **calidad de señal**, no como
resultado de trading directo.

### **6.5. Función para calcular**

In [ ]:
def compute_opportunity_filter_metrics(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    delta_op: float,
    theta: float,
) -> dict:
    """
    Calcula métricas económicas simples para evaluar al modelo como
    filtro de oportunidad durante la ventana de gestación.

    No representa PnL real. Mide calidad de señal y capacidad de anticipar
    movimientos explotables.

    Parámetros
    ----------
    y_true : np.ndarray
        Delta real en puntos (n_samples, seq_len) o (n_samples, seq_len, 1)
    y_pred : np.ndarray
        Delta predicho en puntos (n_samples, seq_len) o (n_samples, seq_len, 1)
    delta_op : float
        Umbral económico que define un movimiento explotable
    theta : float
        Umbral mínimo de predicción para generar señal

    Retorna
    -------
    metrics : dict
        Métricas de filtro económico
    """

    # ------------------------------------------------------------
    # 1) Normalización de shapes
    # ------------------------------------------------------------
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    if y_true.ndim == 3:
        y_true = y_true.squeeze(-1)
    if y_pred.ndim == 3:
        y_pred = y_pred.squeeze(-1)

    assert y_true.shape == y_pred.shape, "Shapes incompatibles"

    # ------------------------------------------------------------
    # 2) Usamos el último paso como referencia económica
    #    (movimiento a h minutos)
    # ------------------------------------------------------------
    y_true_last = y_true[:, -1]
    y_pred_last = y_pred[:, -1]

    # ------------------------------------------------------------
    # 3) Definición de eventos reales (oportunidad económica)
    # ------------------------------------------------------------
    real_opportunity = np.abs(y_true_last) >= delta_op

    # ------------------------------------------------------------
    # 4) Definición de señales del modelo
    # ------------------------------------------------------------
    model_signal = np.abs(y_pred_last) >= theta

    # Señal correcta: magnitud suficiente y signo correcto
    correct_signal = (
        model_signal
        & real_opportunity
        & (np.sign(y_true_last) == np.sign(y_pred_last))
    )

    # ------------------------------------------------------------
    # 5) Métricas económicas como filtro
    # ------------------------------------------------------------

    # Recall: ¿cuántas oportunidades reales fueron detectadas?
    opportunity_recall = (
        correct_signal.sum() / real_opportunity.sum()
        if real_opportunity.sum() > 0 else 0.0
    )

    # Precision: ¿cuántas señales del modelo eran oportunidades reales?
    precision = (
        correct_signal.sum() / model_signal.sum()
        if model_signal.sum() > 0 else 0.0
    )

    # Coverage: qué proporción del tiempo el modelo marca oportunidad
    coverage = model_signal.mean()

    metrics = {
        "Opportunity_Recall": float(opportunity_recall),
        "Precision": float(precision),
        "Coverage": float(coverage),
    }

    return metrics

In [ ]:
#Como se usa:

econ_metrics = compute_opportunity_filter_metrics(
    y_true=y_true,
    y_pred=y_pred,
    delta_op=delta_target_p70,
    theta=20.0,
)

## **7. Arquitectura de notebooks**

# Stage_07_XX – <MODEL_NAME> (Seq2Seq 29x8 → 29x1)

## 0) Objetivo de la notebook
- Qué modelo se entrena y por qué.
- Qué horizonte aplica (h=60 o h=90).
- Qué artefactos produce para Stage_08.

---

## 1) Setup
### 1.1 Imports
- numpy, pandas, torch/keras (según modelo)
- utilidades comunes: `compute_seq2seq_metrics`, `compute_opportunity_filter_metrics`

### 1.2 Reproducibilidad
- seeds (numpy / torch / random)
- device (cpu/cuda)
- flags de determinismo si aplica

---

## 2) Configuración (parámetros)
- `HORIZON = 60 | 90`
- `SEQ_LEN = 29`
- `N_FEATURES = 8`
- hiperparámetros del modelo
- `theta` (umbral señal) y `delta_op` (umbral oportunidad)

---

## 3) Carga de datos (inputs del pipeline)
- cargar `windows_{split}_{h}.npz` (train/valid/test)
- verificar shapes y dtypes
- sanity checks (NaN, rangos, conteos)

---

## 4) Dataloaders / batching
- dataset + dataloader
- definición clara de:
  - `X: (batch, 29, 8)`
  - `Y: (batch, 29, 1)` o `(batch, 29)`

---

## 5) Definición del modelo
- arquitectura
- función de pérdida (MSE / MAE)
- optimizer
- scheduler (opcional)

---

## 6) Entrenamiento
- loop epochs
- early stopping (por valid loss)
- logging mínimo:
  - train_loss, valid_loss por epoch

---

## 7) Evaluación (OOS)
- inferencia sobre valid y test
- cálculo métricas ML:
  - MAE, RMSE, DA_last (y R2 opcional)
- cálculo métricas económicas (filtro):
  - Precision, Opportunity_Recall, Coverage

---

## 8) Guardado de artefactos (para Stage_08)
Guardar en una estructura estándar por modelo/horizonte:

- `reports/stage_07/<h>/<model_name>/metrics_ml.json`
- `reports/stage_07/<h>/<model_name>/metrics_econ.json`
- `reports/stage_07/<h>/<model_name>/pred_test.npz`
  - y_true, y_pred (test)
- `models/stage_07/<h>/<model_name>/model.*` (pesos / joblib / pt)

---

## 9) Resumen final
- tabla corta con métricas principales
- notas de entrenamiento (tiempo, convergencia, issues)